# Ablation and steering experiments
We're going to
- Load the data for helix face value bins on train sequences
- Load all of our test proteins
- Iterate through our test set by randomly picking residues, then doing three different steering experiments:
    1. Mask residue. On layer 18, subtract the mean activation for a residue's current angle, then add the global mean back
        - Controls: subtract mean for random activation instead of bin's own, subtract random vector of same norm
    2. Fit OLS on mean activations, subtract the OLS fitted activation for current value then add back the fitted value for the global mean
        - Controls: subtract fitted activation of random value instead of residue's own, subtract random vector of same norm
    3. Same as above but fit a + b \cos \theta + c \sin \theta instead of OLS
- Endpoints:
    -   Log likelihood of the correct aka. original aminoacid
    -   Top-1 accuracy (is the top prediction changing ?)
    -   95% confidence intervals from each, computed from deciles for effect 

In [1]:
import hashlib

import numpy as np
import polars as pl
import torch
from transformers import AutoTokenizer, EsmForMaskedLM

from biomanifolds import config
from biomanifolds.datasets import swissprot
from biomanifolds.labels import af2_structure_labels

LAYER = 18
MODEL = "facebook/esm2_t30_150M_UR50D"
N_RESIDUES = 10000
rng = np.random.default_rng(0)

# Bin sums and counts of helix face on the train set (from notebook 2): bin k is [k - 180, k - 179) degrees
bin_files = {name: config.data_path("label_bins") / f"train_{name}_L{LAYER}.npy" for name in [
    "helix_face_1deg_bin_sums", "helix_face_1deg_bin_counts"]}
assert all(path.exists() for path in bin_files.values()), "no bin files found, run notebook 2 first"
helix_sums, helix_counts = (np.load(path) for path in bin_files.values())

# ESM-2 with its amino-acid head, to read the log-likelihood of the masked residue
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL, attn_implementation="sdpa").eval().to(device)


# The sampled test proteins (the same 10% MD5 rule as notebook 2)
def in_sample(accession):
    return int(hashlib.md5(accession.encode()).hexdigest()[:8], 16) / 16**8 < 0.10

sequences = pl.read_parquet(swissprot.swissprot_dir() / "swissprot.parquet", columns=["accession", "sequence"])
splits = pl.read_parquet(swissprot.swissprot_dir() / "splits" / "splits.parquet", columns=["accession", "split"])
test_proteins = (
    sequences.join(splits.filter(pl.col("split") == "test"), on="accession")
    .filter(pl.col("accession").map_elements(in_sample, return_dtype=pl.Boolean))
    .select("accession", "sequence")
    .sort("accession")
)
test_proteins

(null): No such file or directory


(null): No such file or directory


Loading weights:   0%|          | 0/491 [00:00<?, ?it/s]

accession,sequence
str,str
"""A0A023PZF5""","""MQNIFYRSAYVHTVCPARLEYYLLLPLLLY…"
"""A0A072VMJ3""","""MGFDNPRSERFEDDPEISKIPTTSGVKVKY…"
"""A0A075TXZ8""","""MYSTFSANFDTTIDASRDRSATPVTAPRPK…"
"""A0A077Y877""","""MHQDNSMRRAINQSRNGGSDSCDINNDREH…"
"""A0A084B9Z3""","""MSTMAKSPEANNLHQDVIAQFPILNGYTHT…"
…,…
"""V9TSX0""","""MRKWLIFLLIAAVAGLSACSTSGNTTVSDD…"
"""W0TEF9""","""MLRSTLSNWREYLTPITHTSTFETTGQLTP…"
"""W1QDU5""","""MSNSPPSSSSSGKGLSIPRKLALNAINEIQ…"


## Fitting OLS on activations, as well as activations transformed by harmonic functions

In [2]:
theta = np.deg2rad(np.arange(360) - 179.5)  # centre of each 1-degree bin, in radians
H = helix_sums / helix_counts[:, None]  # (360, 640) mean activation per bin
global_mean = helix_sums.sum(0) / helix_counts.sum()  # mu: mean activation over all train helix residues
weights = np.sqrt(helix_counts)[:, None]  # least squares on the bin means, weighted by the residues per bin

# Mean activation for a specific value of the latent
def bin_mean_activation(t):
    return H[int(np.floor(np.rad2deg(t) + 180)) % 360]

# OLS on linear inputs
X = np.column_stack([np.ones(360), theta])
B_linear = np.linalg.lstsq(X * weights, H * weights, rcond=None)[0]
def linear_activation(t):
    return np.array([1.0, t]) @ B_linear

# OLS on inputs transformed by harmonic functions
X = np.column_stack([np.ones(360), np.cos(theta), np.sin(theta)])
B_harmonic = np.linalg.lstsq(X * weights, H * weights, rcond=None)[0]
def harmonic_activation(t):
    return np.array([1.0, np.cos(t), np.sin(t)]) @ B_harmonic

# Object containing the fitted activations for each angle according to each method
activation_at_angle = {"bin_means": bin_mean_activation, "linear": linear_activation, "harmonic": harmonic_activation}

# weighted R^2 of the two fits on the bin means
total = (helix_counts[:, None] * (H - global_mean) ** 2).sum()
pl.DataFrame([
    {"fit": name, "r2": 1 - (helix_counts[:, None] * (H - np.stack([activation(t) for t in theta])) ** 2).sum() / total}
    for name, activation in [("linear", linear_activation), ("harmonic", harmonic_activation)]
])

fit,r2
str,f64
"""linear""",0.152217
"""harmonic""",0.870341


## Randomly sampling 10k residues from the test dataset

In [3]:
residues = (
    af2_structure_labels.load_labels(["accession", "i", "helix_face"])
    .filter(pl.col("helix_face").is_not_null())
    .join(test_proteins.lazy(), on="accession")
    .collect(engine="streaming")
    .sort("accession", "i")  # fixed order, so that the sample is reproducible
    .sample(N_RESIDUES, seed=0)
)
residues

accession,i,helix_face,sequence
str,i32,f32,str
"""E4V6L3""",265,-136.852249,"""MAVHHKRAGSHRGRGSKAGRSRGRKFETSR…"
"""P09207""",25,-154.556427,"""MREIVHLQIGQCGNQIGAKFWEVISDEHGI…"
"""O74088""",45,137.638168,"""MVKSKVKKVEKGKEGEEKRSTYVLLKKVLI…"
"""A0L1N4""",444,-40.564983,"""MFGKLLTKVFGSRNDRTLKGLQKVVNKINA…"
"""P9WGD0""",245,109.674538,"""MPKISSRDGGRPAQRTVNPIIVTRRGKIAR…"
…,…,…,…
"""P46577""",120,71.256599,"""MAEKAENLPSSSAEASEEPSPQTGPNVNQK…"
"""Q80W92""",203,130.537308,"""MNPEKDFAPLTPNIVRALNDKLYEKRKVAA…"
"""B5F683""",77,1.851507,"""MSVPLILTLLAGAATFIGAFLGVLGQKPSN…"


## Editing activations
For each randomly sampled residue, ablate the fitted latent vector for its angle according to each different method, measure the effect on logits and the top-predicted aminoacid.

In [4]:
def run_layers(h, start, end):
    """Run ESM layers start+1 .. end on the hidden states h (batch, tokens, 640)."""
    rotary = model.esm.rotary_embeddings(h, torch.arange(h.shape[1], device=h.device)[None])  # rotary position embeddings
    for block in model.esm.encoder.layer[start:end]:
        h = block(h, position_embeddings=rotary)
    return h


results = []
for accession, i, face, sequence in residues.select("accession", "i", "helix_face", "sequence").iter_rows():
    tokens = tokenizer([sequence], return_tensors="pt").to(device)
    position = i + 1  # token 0 is <cls>
    original = tokens["input_ids"][0, position].item()
    tokens["input_ids"][0, position] = tokenizer.mask_token_id

    # the change added to h at the masked position: mu - f(theta)
    own, random_angle = np.deg2rad(face), rng.uniform(-np.pi, np.pi)
    deltas = {"none": np.zeros(H.shape[1])}
    for method, activation in activation_at_angle.items():
        deltas[method] = global_mean - activation(own)
        deltas[f"{method} | random angle"] = global_mean - activation(random_angle)
        random_direction = rng.standard_normal(H.shape[1])
        deltas[f"{method} | random vector"] = random_direction / np.linalg.norm(random_direction) * np.linalg.norm(deltas[method])

    # layers 1..LAYER once, then the edits at the masked position, then layers LAYER+1..30 for every edit at once
    with torch.no_grad():
        h = model.esm.embeddings(input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"])
        h = run_layers(h, 0, LAYER).repeat(len(deltas), 1, 1)  # one copy per edit
        h[:, position] += torch.tensor(np.stack(list(deltas.values())), dtype=torch.float32, device=device)
        h = run_layers(h, LAYER, len(model.esm.encoder.layer))
        log_probs = model.lm_head(model.esm.encoder.emb_layer_norm_after(h))[:, position].log_softmax(-1)  # (edits, vocabulary)
    for name, lp in zip(deltas, log_probs):
        results.append({"accession": accession, "i": i, "edit": name,
                        "log_likelihood": lp[original].item(), "top1_correct": lp.argmax().item() == original})

results = pl.DataFrame(results)
results

accession,i,edit,log_likelihood,top1_correct
str,i64,str,f64,bool
"""E4V6L3""",265,"""none""",-0.924206,true
"""E4V6L3""",265,"""bin_means""",-0.957876,true
"""E4V6L3""",265,"""bin_means | random angle""",-0.851416,true
"""E4V6L3""",265,"""bin_means | random vector""",-1.057738,true
"""E4V6L3""",265,"""linear""",-0.99553,true
…,…,…,…,…
"""B8AFI3""",469,"""linear | random angle""",-0.941087,true
"""B8AFI3""",469,"""linear | random vector""",-0.941004,true
"""B8AFI3""",469,"""harmonic""",-0.952684,true


## Effects
Change of each endpoint against the unedited pass of the same residue, with bootstrap 95% confidence intervals of the mean.

In [ ]:
# Confidence intervals from deciles
def bootstrap_ci(x, n=10_000):
    means = rng.choice(x, (n, len(x))).mean(1)
    return np.percentile(means, [2.5, 97.5])

# Print out results with confidence intervals
unedited = results.filter(pl.col("edit") == "none").select("accession", "i", ll_unedited="log_likelihood", top1_unedited="top1_correct")
effects = results.join(unedited, on=["accession", "i"]).with_columns(
    delta_ll=pl.col("log_likelihood") - pl.col("ll_unedited"),
    delta_top1=pl.col("top1_correct").cast(pl.Float64) - pl.col("top1_unedited").cast(pl.Float64),
)
summary = []
for (name,), group in effects.group_by("edit", maintain_order=True):
    ll_low, ll_high = bootstrap_ci(group["delta_ll"].to_numpy())
    top1_low, top1_high = bootstrap_ci(group["delta_top1"].to_numpy())
    summary.append({"edit": name, "mean_delta_ll": group["delta_ll"].mean(), "ll_ci_low": ll_low, "ll_ci_high": ll_high,
                    "top1_accuracy": group["top1_correct"].mean(), "mean_delta_top1": group["delta_top1"].mean(),
                    "top1_ci_low": top1_low, "top1_ci_high": top1_high})
pl.DataFrame(summary)

edit,mean_delta_ll,ll_ci_low,ll_ci_high,top1_accuracy,mean_delta_top1,top1_ci_low,top1_ci_high
str,f64,f64,f64,f64,f64,f64,f64
"""none""",0.0,0.0,0.0,0.4379,0.0,0.0,0.0
"""bin_means""",-0.000375,-0.001076,0.000337,0.4377,-0.0002,-0.002,0.0017
"""bin_means | random angle""",-0.000814,-0.00152,-0.000125,0.4387,0.0008,-0.001,0.0026
"""bin_means | random vector""",-0.000883,-0.001565,-0.00021,0.4371,-0.0008,-0.0025,0.001
"""linear""",-3.2427e-7,-0.000261,0.000262,0.4376,-0.0003,-0.0014,0.0008
"""linear | random angle""",-0.000324,-0.000606,-0.00005,0.438,0.0001,-0.0009,0.0011
"""linear | random vector""",0.000132,-0.000122,0.00038,0.4367,-0.0012,-0.0023,-0.0002
"""harmonic""",-0.000358,-0.000988,0.000282,0.4383,0.0004,-0.0015,0.0023
"""harmonic | random angle""",-0.000765,-0.001413,-0.000118,0.4389,0.001,-0.0007,0.0028
